# Step 2 - Generate audio book with Azure Speech

In [ ]:
%pip install openai
%pip install python-dotenv
%pip install pydub
%pip install azure-core
%pip install azure-cognitiveservices-speech
%pip install azure-ai-documentintelligence
%pip install markd
%pip install semantic-kernel

In [15]:
# Add azure open ai endpoint
import os,re
from dotenv import load_dotenv

load_dotenv(dotenv_path='../keys.env')
azure_openai_endpoint = os.getenv('AZURE_OPENAI_ENDPOINT')
azure_openai_key = os.getenv('AZURE_OPENAI_KEY')
azure_speech_key = os.getenv('AZURE_SPEECH_KEY')
azure_speech_region = os.getenv('AZURE_SPEECH_REGION')
azure_destination_container_url = os.environ['AZURE_DESTINATION_CONTAINER_URL']

folder_path = '../data/hachette'  # Adjust this path as needed
mdFileNameAddition = "_full_text"

# https://learn.microsoft.com/en-us/azure/ai-services/speech-service/high-definition-voices#how-to-use-azure-ai-speech-hd-voices
prosodyTitleRate = '-10.00%'
prosodyTitlePitch = '-8.00%'
prosodyTextRate = '-8.00%'
prosodyTextPitch = '00.00%'
voiceTemperature = '0.8'


In [ ]:
def convert_markdown_to_ssml(input_file, output_file, language='en-GB'):
    """
    Converts a markdown file to SSML format for Azure Speech Services.
    Handles titles separately and removes problematic line breaks.
    """
    
    # Set the voice based on the detected language (you might need to adjust voice names)

    if language == 'fr':
        voice = "fr-FR-Vivienne:DragonHDLatestNeural"  # Example French voice
    elif language == 'en-GB':
        # voice = "en-US-Ava3:DragonHDLatestNeural"  # Example English voice - preview
        # voice = "en-US-Andrew3:DragonHDLatestNeural"  # Example English voice - preview
        # voice = "en-US-Emma2:DragonHDLatestNeural" # GA (cool)
        # voice = "en-US-Andrew2:DragonHDLatestNeural"  # GA
        # voice = "en-US-Brian:DragonHDLatestNeural" # GA
        # voice = "es-ES-Ximena:DragonHDLatestNeural" # GA interesting voice for grandma style ? :-)

        # voice = "en-US-Alloy:DragonHDLatestNeural" # preview
        # voice = "en-US-Aria:DragonHDLatestNeural" # preview
        # voice = "en-US-Jenny:DragonHDLatestNeural" # preview
        # voice = "en-US-Nova:DragonHDLatestNeural" # preview
        # voice = "en-US-Phoebe:DragonHDLatestNeural" # preview
        # voice = "en-US-Serena:DragonHDLatestNeural" # preview
        
        voice = "en-US-MultiTalker-Ava-Andrew:DragonHDLatestNeural" # preview, set multiVoiceMode = True
    else:
        voice = "en-US-Andrew3:DragonHDLatestNeural"  # Default to English
    
    multiTalkerMode = True # set to True to use multi voice mode (Ava and Andrew) - preview

    ''' HD Neural GA Voices
    de-DE-Florian:DragonHDLatestNeural
    de-DE-Seraphina:DragonHDLatestNeural
    en-US-Adam:DragonHDLatestNeural
    en-US-Andrew:DragonHDLatestNeural
    en-US-Andrew2:DragonHDLatestNeural
    en-US-Ava:DragonHDLatestNeural
    en-US-Brian:DragonHDLatestNeural
    en-US-Davis:DragonHDLatestNeural
    en-US-Emma:DragonHDLatestNeural
    en-US-Emma2:DragonHDLatestNeural
    en-US-Steffan:DragonHDLatestNeural
    es-ES-Tristan:DragonHDLatestNeural
    es-ES-Ximena:DragonHDLatestNeural
    fr-FR-Remy:DragonHDLatestNeural
    fr-FR-Vivienne:DragonHDLatestNeural
    ja-JP-Masaru:DragonHDLatestNeural
    ja-JP-Nanami:DragonHDLatestNeural
    zh-CN-Xiaochen:DragonHDLatestNeural
    zh-CN-Yunfan:DragonHDLatestNeural
'''

    
    try:
        # Read the markdown file
        with open(input_file, 'r', encoding='utf-8') as f:
            content = f.read()

        # Remove PageBreaks
        lines = content.split('\n')
        i=0
        lines2 = list()
        # Loop through all the lines in the file
        while i < len(lines):
            line = lines[i].strip()
            if line == '<!-- PageBreak -->':
                # Remove the empty lines before and after
                if lines[i-1].strip() =='':
                    lines2.pop()
                if i < len(lines)-1 and lines[i+1].strip() =='':
                    i=i+1
            elif not line.startswith("<!-- PageNumber="):
                lines2.append(escape(line))
            i += 1
        
        # Remove last line if starts with #
        # This is because some books have two lines for new chapter and we got the first one at the end of the chapter file
        if lines2[-1].startswith('#'):
            lines2.pop()

        # tag titles and ignore Notes
        lines = lines2.copy()
        i=0
        lines2.clear()
        while i < len(lines):
            line = lines[i].strip()

            if line.startswith('\\* '):
                # ignore all lines as these are notes
                break

            if line.startswith('#'):
                if " notes" in line.lower():
                    # Ignore all lines with Notes
                    break
                title_level = len(re.match(r'^(#+)', line).group(1))
                title_text = line.replace('#', '').strip()
                # Replace the line with an empty string
                lines2.append('#'+title_text)
          
            else:
                lines2.append(line)
            i += 1
            
        content = '\n'.join(lines2)
        
        # Clean the text - remove markdown formatting
        content = re.sub(r'\*\*(.*?)\*\*', r'\1', content)  # Remove bold
        content = re.sub(r'\*(.*?)\*', r'\1', content)      # Remove italic
        content = re.sub(r'!\[(.*?)\]\(.*?\)', '', content) # Remove images
        content = re.sub(r'\[(.*?)\]\(.*?\)', r'\1', content) # Convert links to text
        content.replace('<figure>','') # remove firure syntax
        content.replace('</figure>','') # remove figure syntax
   
        # Construct SSML
        # subset supported for OpenAI voices https://learn.microsoft.com/en-us/azure/ai-services/speech-service/openai-voices#ssml-elements-supported-by-openai-text-to-speech-voices-in-azure-ai-speech
        # subset for Neural HD https://learn.microsoft.com/en-us/azure/ai-services/speech-service/high-definition-voices#supported-and-unsupported-ssml-elements-for-azure-ai-speech-hd-voices
        if multiTalkerMode:
            ssml = f'''<speak version="1.0" xmlns="http://www.w3.org/2001/10/synthesis" xmlns:mstts="http://www.w3.org/2001/mstts" xml:lang="{language}" parameters='temperature={voiceTemperature}'>\n<voice name="{voice}"><mstts:dialog>\n'''
        else:
            ssml = f'''<speak version="1.0" xmlns="http://www.w3.org/2001/10/synthesis" xml:lang="{language}" parameters='temperature={voiceTemperature}'>\n<voice name="{voice}">\n'''
          
        # Add paragraphs with appropriate breaks
        paragraphs = re.split(r'\n\s*\n', content.strip())
        for paragraph in paragraphs:
            if paragraph.strip():
                # Clean paragraph text
                cleaned_paragraph = paragraph.replace('\n', ' ').strip()
                if cleaned_paragraph:
                    if cleaned_paragraph.startswith('#'):
                        if multiTalkerMode:
                            ssml += f'<mstts:turn speaker="andrew"><prosody rate="{prosodyTitleRate}" pitch="{prosodyTitlePitch}"><p>{cleaned_paragraph.lstrip('#')}</p></prosody></mstts:turn>\n'
                        else:
                            ssml += f'<prosody rate="{prosodyTitleRate}" pitch="{prosodyTitlePitch}"><p>{cleaned_paragraph.lstrip('#')}</p></prosody>\n'
                    else:
                        if multiTalkerMode:
                            ssml += f'<mstts:turn speaker="ava"><prosody rate="{prosodyTextRate}" pitch="{prosodyTextPitch}"><p>{cleaned_paragraph}</p></prosody></mstts:turn>\n'
                        else:
                            ssml += f'<prosody rate="{prosodyTextRate}" pitch="{prosodyTextPitch}"><p>{cleaned_paragraph}</p></prosody>\n'
        
        if multiTalkerMode:
            ssml += '</mstts:dialog></voice>\n</speak>'
        else:
            ssml += '</voice>\n</speak>'
        
        # Write SSML to output file
        with open(output_file, 'w', encoding='utf-8') as f:
            f.write(ssml)
            
        return True
    except Exception as e:
        print(f"Error converting {input_file} to SSML: {e}")
        return False
    
def escape( str_xml: str ):
    str_xml = str_xml.replace("&", "&amp;")
    str_xml = str_xml.replace("<", "&lt;")
    str_xml = str_xml.replace(">", "&gt;")
    #str_xml = str_xml.replace("\"", "&quot;")
    #str_xml = str_xml.replace("'", "&apos;")
    return str_xml

def process_markdown_to_ssml(directory):
    """
    Process all markdown files in a directory and convert them to SSML files.
    """
    converted_files = []
    
    # Look for chapter directories from split_markdown_by_chapters function
    for root, dirs, files in os.walk(directory):
        for filename in files:
            if filename.endswith(".md") and not filename.endswith(mdFileNameAddition+".md") and not filename.startswith("output-"):
                md_file_path = os.path.join(root, filename)
                ssml_file_path = os.path.join(root, f"{os.path.splitext(filename)[0]}-speech.ssml")
                
                print(f"Converting {md_file_path} to SSML...")
                if convert_markdown_to_ssml(md_file_path, ssml_file_path):
                    converted_files.append(ssml_file_path)
                    print(f"Successfully converted to {ssml_file_path}")
                else:
                    print(f"Failed to convert {md_file_path}")
    
    return converted_files

# Process markdown files in the root directory
converted_ssml_files = process_markdown_to_ssml(folder_path)
print(f"Converted {len(converted_ssml_files)} markdown files to SSML")

## Generate audio book 

In [ ]:
output_folder = "audio_output"  # Temporary folder for page-level audio

speech_region = azure_speech_region
speech_key = azure_speech_key

processOnlyFirstChapter = False

priceTextToSpeechNeuralHDFor1M = 30 # USD for 1M Characters

class SpeechJobReport:
    jobid = None
    audiofolder = None
    relativepath = None
    filename = None
    status = None
    audiolocalfolder = None

    def __init__(self, filename, relativepath):
        self.filename = filename
        self.relativepath = relativepath

    def add_jobid(self, jobid):
        self.jobid = jobid

    def add_audio_folder(self, audiofolder):
        self.audiofolder = audiofolder

    def add_audio_local_folder(self, audiolocalfolder):
        self.audiolocalfolder = audiolocalfolder

    def add_status(self, status):
        self.status = status

In [ ]:
# BATCH MODE for audio generation using the REST API

import json
import os
import time
import uuid
from pathlib import Path
import datetime
import requests
from typing import List
import urllib.request
import zipfile
import json

API_VERSION = "2024-04-01"

def _create_job_id():
    # the job ID must be unique in current speech resource
    # you can use a GUID or a self-increasing number
    return uuid.uuid4()

def _authenticate():
    return {'Ocp-Apim-Subscription-Key': speech_key}

def submit_synthesis(job_id: str, text : str) -> bool:
    url = f'https://{speech_region}.api.cognitive.microsoft.com/texttospeech/batchsyntheses/{job_id}?api-version={API_VERSION}'
    header = {
        'Content-Type': 'application/json'
    }
    header.update(_authenticate())

    paragraphs = re.split(r'\n', text.strip())
    dataSSML = []
    
    # enumerate the paragraphs and copy the header and footer to each paragraph. We do this to split the SSML into the body of REST call
    for i in range(2, len(paragraphs)-2):
            netext =paragraphs[0]+ paragraphs[1]+ paragraphs[i].strip()+paragraphs[len(paragraphs)-2]+ paragraphs[len(paragraphs)-1]
            dataSSML.append(netext)

    # create an json array of content with the first 2 paragraphs and the rest of the paragraphs
    splitSSML = [{"content": ssml} for ssml in dataSSML]

    payload = {
        "inputKind": "SSML", # or PlainText SSML
        #'synthesisConfig': {
        #    "voice": "en-US-Ava3:DragonHDLatestNeural" #"en-US-AvaMultilingualNeural",
        #},
        # Replace with your custom voice name and deployment ID if you want to use custom voice.
        # Multiple voices are supported, the mixture of custom voices and platform voices is allowed.
        # Invalid voice name or deployment ID will be rejected.
        #'customVoices': {
            # "YOUR_CUSTOM_VOICE_NAME": "YOUR_CUSTOM_VOICE_ID"
        #},
        #"inputs": [
        #    {
        #        "content": text
        #    },
        #],
        "inputs": splitSSML,
         
        "BatchSynthesisOutputs": {
            "result" :"result",
            "summary": "summary",
        },
        "properties": {
            # all properties doc https://learn.microsoft.com/en-us/rest/api/batchtexttospeech/batch-syntheses/create?view=rest-batchtexttospeech-2024-04-01&tabs=HTTP#batchsynthesisproperties

            # see https://learn.microsoft.com/en-us/azure/ai-services/speech-service/rest-text-to-speech?tabs=nonstreaming#audio-outputs
            "outputFormat": "riff-48khz-16bit-mono-pcm", # best quality
            # "decompressOutputFiles": True,
            "concatenateResult": True,
            # "destinationContainerUrl": azure_destination_container_url
        },
    }
    response = requests.put(url, json.dumps(payload), headers=header)
    if response.status_code < 400:
        # print(f'Batch synthesis job submitted successfully ({response.json()["id"]})')
        return True
    else:
        print(f'Failed to submit batch synthesis job: [{response.status_code}], {response.text}')
        return False

def get_synthesis(job_id: str) -> str:
    url = f'https://{speech_region}.api.cognitive.microsoft.com/texttospeech/batchsyntheses/{job_id}?api-version={API_VERSION}'
    header = _authenticate()
    response = requests.get(url, headers=header)
    if response.status_code < 400:
        # print('Get batch synthesis job successfully')
        # print(response.json())
        return response.json()
    else:
        print(f'Failed to get batch synthesis job: {response.text}')
        data = {}
        data['status'] = 'Unknown'
        
        return data

def get_synthesis_audio_folder(job_id: str):
    url = f'https://{speech_region}.api.cognitive.microsoft.com/texttospeech/batchsyntheses/{job_id}?api-version={API_VERSION}'
    header = _authenticate()
    response = requests.get(url, headers=header)
    vv = response.json()['outputs']['result']
    if response.status_code < 400:
        return response.json()['outputs']['result']
    else:
        print(f'Failed to get batch synthesis job: {response.text}')

def list_synthesis_jobs(skip: int = 0, max_page_size: int = 100):
    """List all batch synthesis jobs in the subscription"""
    url = f'https://{speech_region}.api.cognitive.microsoft.com/texttospeech/batchsyntheses?api-version={API_VERSION}&skip={skip}&maxpagesize={max_page_size}'
    header = _authenticate()
    response = requests.get(url, headers=header)
    if response.status_code < 400:
        print(f'List batch synthesis jobs successfully, got {len(response.json()["values"])} jobs')
        print(response.json())
    else:
        print(f'Failed to list batch synthesis jobs: {response.text}')


def file_should_be_processed(filename: str) -> bool:
    return filename.endswith("-speech.ssml") and (not processOnlyFirstChapter or filename.startswith("0")) and not filename.startswith("output-")

def process_chapter_audio(chapter_dir) -> list[SpeechJobReport]:
    """
    Processes each chapter file in the given directory to generate audio.
    """
    SpeechJobReportList = []

    chapter_job_id = []
    for filename in os.listdir(chapter_dir):
        if file_should_be_processed(filename):
            chapter_file_path = os.path.join(chapter_dir, filename)
    
            relativepath = os.path.relpath(chapter_dir, parent_path)
            mySpeechJobReport = SpeechJobReport(filename, relativepath)

            try:
                with open(chapter_file_path, "r", encoding="utf-8") as chapter_file:
                    chapter_content = chapter_file.read()
                job_id = _create_job_id()
                print(f"Generate audio from ssml file: {filename}, jobid: {job_id}")
                if submit_synthesis(job_id, chapter_content):
                    chapter_job_id.append(job_id)
                    mySpeechJobReport.add_jobid(job_id)
                    SpeechJobReportList.append(mySpeechJobReport)
                else:
                    print(f"Failed to generate audio for {filename}")
            except Exception as e:
                print(f"Error processing {filename}: {e}")
    
    statusOfAllJobs = {}
    finished = {}
    for job_id in chapter_job_id:
        statusOfAllJobs[job_id] = 'None'
        finished[job_id] = False


    try:
        # while not all statusOfAllJobs items are set 'Succeeded' or 'Failed'
        while True:
            for job_id in chapter_job_id:
                result = get_synthesis(job_id)
                status = result['status']
                                
                if status == 'Succeeded' and statusOfAllJobs[job_id] != 'Succeeded':
                    # print(f'Batch synthesis job {job_id} succeeded ({datetime.now().strftime('%H:%M:%S')})')
                    finished[job_id] = True
                    for sjrl in SpeechJobReportList:
                        if sjrl.jobid == job_id:
                            break
                    sjrl.add_audio_folder(get_synthesis_audio_folder(job_id))
                    sjrl.add_status(status)
                    
                elif status == 'Failed' and statusOfAllJobs[job_id] != 'Failed':
                    # print(f'Batch synthesis job {job_id} failed ({datetime.now().strftime('%H:%M:%S')})')
                    # print(result)
                    finished[job_id] = True
                    for sjrl in SpeechJobReportList:
                        if sjrl.jobid == job_id:
                            break
                    sjrl.add_audio_folder(get_synthesis_audio_folder(job_id))
                    sjrl.add_status(status)
                    
                elif status != 'Succeeded' and status != 'Failed':
                    finished[job_id] = False
                    
                statusOfAllJobs[job_id] = status

            if all(finished.values()):
                break
            print(f'Processing job(s): total : {len(finished)}, finished : {list(finished.values()).count(True)}, updated on {datetime.datetime.now().strftime('%H:%M:%S')}', end='\r')
            time.sleep(15) 
        
    except Exception as e:
        print(f"Error during speech synthesis: {e}")
        return False
    
    return SpeechJobReportList

stringUniqueDateTime = datetime.datetime.now().strftime('%Y_%m_%d_%H_%M_%S')
output_folder_unique = f"{output_folder}_{stringUniqueDateTime}" 

# Create temporary folder if it doesn't exist
parent_path = os.path.split(folder_path)[0]
if not os.path.exists(f"{parent_path}\\{output_folder_unique}"):
    os.makedirs(f"{parent_path}\\{output_folder_unique}")


job_reports = []

# Create a MD file to store the output data
parent_path = os.path.split(folder_path)[0]
mdfileoutput = f"{parent_path}\\output_speech_{stringUniqueDateTime}.md"

def copy_ssml_to_dest(chapter_dir, report, destination_folder):
    ssmlfile = os.path.join(chapter_dir, report.filename)
    ssmlfiledest = os.path.join(destination_folder, report.filename)
            # copy the file to the destination folder
    with open(ssmlfile, 'rb') as fsrc:
        with open(ssmlfiledest, 'wb') as fdst:
            fdst.write(fsrc.read())

with open(mdfileoutput, 'ab+') as mdfile:
    mdfile.write('# Azure Speech Voice outputs\n\n'.encode())
    mdfile.write(f'Generated on {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n'.encode())
    mdfile.write("|book|ssml file|job id|wav duration|billable char|price (usd)|online results|local results|\n".encode())
    mdfile.write("|-|-|-|-|-|-|-|-|\n".encode())
   
    # Get full path of the folder
    folder_path = os.path.abspath(folder_path)

    def download_and_unzip_report(report):
        # Get parent directory
        folderdestroot =  f"{parent_path}\\{output_folder_unique}\\{report.relativepath}"
        zipfilename =  f"{folderdestroot}\\{report.filename}-zip.zip"
        os.makedirs(folderdestroot, exist_ok=True)
        urllib.request.urlretrieve(report.audiofolder, zipfilename)
        # unzip the file to the same directory
        destination_folder = f"{folderdestroot}\\{report.filename}"
        os.makedirs(destination_folder, exist_ok=True)
        zip_file = zipfile.ZipFile(zipfilename)
        zip_file.extractall(destination_folder)
        return destination_folder

    for subdir, dirs, files in os.walk(folder_path):
        print(f"Reading directory: {subdir}")
        # process each chapt files
        chapter_dir = subdir
        jobReports = process_chapter_audio(chapter_dir)
        for report in jobReports:
            # Download the zip file using urllib
            destination_folder = download_and_unzip_report(report)

            # let's copy the ssml to the destination folder
            copy_ssml_to_dest(chapter_dir, report, destination_folder)
           
            report.add_audio_local_folder(destination_folder)
            job_reports.append(report)
            localrelativepathparent = os.path.relpath(destination_folder, parent_path)
            bookname = os.path.basename(os.path.dirname(subdir))

            # Load summary.json file
            # Open the JSON file and load its content
            with open(destination_folder + '/summary.json', 'r') as file1:
                summary = json.load(file1)
                with open(destination_folder + '/0001.debug.json', 'r') as file2:
                    debug = json.load(file2)
                    billcharacters = 0
                    for debuginfo in debug['debugInfo']:
                        billcharacters += int(debuginfo['billableCharacters'])
                    mdfile.write(f"|{bookname} | [{report.filename}](<./{localrelativepathparent.replace("\\", "/")}/{report.filename}>) | {report.jobid}  | {str(datetime.timedelta(seconds=int(summary['results'][0]['properties']['durationInMilliseconds'])//1000))} | {billcharacters} | {round((billcharacters*priceTextToSpeechNeuralHDFor1M)/1000000,3)} | [zip file]({report.audiofolder}) | [folder](<./{localrelativepathparent.replace("\\", "/")}>)|\n".encode())
    
    print(f"Markdown output file saved to <{mdfileoutput}>")

In [ ]:
# If needed, get a specific job info

print(get_synthesis("4e77d8c8-b0fd-4de1-87f2-0d3d47302e12"))
print(get_synthesis("bf0c232e-9a72-42e6-98a6-b34a7e7f4390"))
print(get_synthesis("870d070f-0b03-4c3f-9165-00d061aae1fc"))

In [ ]:
# If needed, delete all jobs

import os
from pathlib import Path
import requests
from typing import List

API_VERSION = "2024-04-01"

def delete_synthesis_jobs(skip: int = 0, max_page_size: int = 100):
    """Delete all batch synthesis jobs in the subscription"""
    url = f'https://{speech_region}.api.cognitive.microsoft.com/texttospeech/batchsyntheses?api-version={API_VERSION}&skip={skip}&maxpagesize={max_page_size}'
    header = _authenticate()
    response = requests.get(url, headers=header)
    if response.status_code < 400:
        for job in response.json()['value']:
            job_id = job['id']
            url = f'https://{speech_region}.api.cognitive.microsoft.com/texttospeech/batchsyntheses/{job_id}?api-version={API_VERSION}'
            header = _authenticate()
            response = requests.delete(url, headers=header)
            if response.status_code < 400:
                print(f'Deleted job: {job_id}')
            else:
                print(f'Failed to delete job {job_id}: {response.text}')
        
    else:
        print(f'Failed to list batch synthesis jobs: {response.text}')

def _authenticate():
    return {'Ocp-Apim-Subscription-Key': speech_key}


delete_synthesis_jobs()

In [ ]:
# write code to send a SSML file to OpenAI using Semantic kernel and asking the model to change the text to add emphasis and breaks

import os
import sys
import semantic_kernel as sk
from semantic_kernel.connectors.ai.open_ai import AzureChatCompletion
from semantic_kernel.prompt_template import PromptTemplateConfig
from semantic_kernel.prompt_template.input_variable import InputVariable
from semantic_kernel.connectors.ai.open_ai import AzureChatPromptExecutionSettings, OpenAIChatPromptExecutionSettings
from semantic_kernel.prompt_template import PromptTemplateConfig
from semantic_kernel.prompt_template.input_variable import InputVariable
from semantic_kernel.functions import KernelArguments

from semantic_kernel.contents import ChatHistory




import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv(dotenv_path='../keys.env')

# Azure OpenAI credentials
azure_openai_endpoint = os.getenv('AZURE_OPENAI_ENDPOINT')
azure_openai_key = os.getenv('AZURE_OPENAI_KEY')
deployment_name = "gpt-4"  # Adjust this to your deployment name

# Initialize Semantic Kernel
kernel = sk.Kernel()
kernel.add_service(
    AzureChatCompletion(
        deployment_name=deployment_name,
        endpoint=azure_openai_endpoint,
        api_key=azure_openai_key,
        service_id= "chat-gpt"
    ),
)

# Define the prompt to instruct the model
prompt = """
You are an expert in SSML (Speech Synthesis Markup Language). 
Given the following SSML content, enhance it by adding appropriate emphasis and breaks to improve text to speech synthesis quality. 
Only modify the text within <p> tags, adding <emphasis> and <break> tags where suitable. 
Do not alter the SSML structure or other tags.

SSML Input:
{{$input}}

Enhanced SSML Output:
"""

# Define the request settings
req_settings = kernel.get_prompt_execution_settings_from_service_id("chat-gpt")
req_settings.max_tokens = 4096
req_settings.temperature = 0.7
req_settings.top_p = 0.8

prompt_template_config = PromptTemplateConfig(
    template=prompt,
    name="chat",
    template_format="semantic-kernel",
    input_variables=[
        InputVariable(name="user_input", description="The user input", is_required=True),
    ],
    execution_settings=req_settings,
)


# Create Semantic Kernel function
ssml_enhancer = kernel.add_function(
    function_name="chat",
    plugin_name="chatPlugin",
    prompt_template_config=prompt_template_config,
)

# Function to read SSML file, enhance it, and save the result
async def enhance_ssml(input_ssml_path, output_ssml_path):
    with open(input_ssml_path, 'r', encoding='utf-8') as file:
        ssml_content = file.read()
  
    arguments = KernelArguments(user_input=ssml_content)
    
    # Run the Semantic Kernel function
    response = await kernel.invoke(ssml_enhancer, input=ssml_content)

    print(f"Output: {response}")
  
    # Save the enhanced SSML to a new file
    with open(output_ssml_path, 'w', encoding='utf-8') as file:
        file.write(str(response))

    print(f"Enhanced SSML saved to {output_ssml_path}")

# Example usage
input_ssml_file = "../data/hachette/Theory of teenagers/A New Theory of Teenagers_ Seven Transform - Christa Santangelo_full_text_chapters/0___INTRODUCTION-speech.ssml"
output_ssml_file = "../data/hachette/Theory of teenagers/A New Theory of Teenagers_ Seven Transform - Christa Santangelo_full_text_chapters/0___INTRODUCTION-speech-enhanced.ssml"

await enhance_ssml(input_ssml_file, output_ssml_file)



Output: <speak version="1.0" xmlns="http://www.w3.org/2001/10/synthesis" xmlns:mstts="http://www.w3.org/2001/mstts" xml:lang="en-US" parameters='temperature=0.8'>
<voice name="en-US-MultiTalker-Ava-Andrew:DragonHDLatestNeural"><mstts:dialog>
<mstts:turn speaker="andrew"><prosody rate="-10.00%" pitch="-8.00%"><p><emphasis level="strong">INTRODUCTION</emphasis></p></prosody></mstts:turn>
<mstts:turn speaker="ava"><prosody rate="-8.00%" pitch="00.00%"><p>There is no greater agony than bearing an untold story inside you.</p></prosody></mstts:turn>
<mstts:turn speaker="ava"><prosody rate="-8.00%" pitch="00.00%"><p><emphasis level="moderate">-Zora Neale Hurston</emphasis></p></prosody></mstts:turn>
<mstts:turn speaker="ava"><prosody rate="-8.00%" pitch="00.00%"><p>I WROTE THIS BOOK. OR DID THIS BOOK WRITE ME? As a psychologist trained at Yale, and an assistant clinical professor at the University of California, San Francisco, I specialize in adolescents and their families. In addition to stu

In [ ]:

def combine_audio_files(input_files, output_file):
    """
    Combines multiple audio files into a single audio file.
    """
    combined_audio = AudioSegment.empty()
    for file in input_files:
        audio = AudioSegment.from_wav(file)
        combined_audio += audio

    combined_audio.export(output_file, format="wav")
    print(f"Combined audio saved to {output_file}")

# Combine the audio files for all chapters
if chapter_audio_files:
    final_output_path = os.path.join(directory, f"{os.path.splitext(filename)[0]}_combined_chapters.wav")
    combine_audio_files(chapter_audio_files, final_output_path)

    # Clean up temporary audio files
    for file in chapter_audio_files:
        os.remove(file)
    print(f"Temporary audio files removed from {output_folder}")
else:
    print("No audio files were created for this PDF.")